# Neural Local Volatility benchmark v4 — shared diagnostic

This is a deliberately slimmer debugging notebook for the **shared neural-LV** model only.

The aim is to answer one question before spending hours on a serious calibration:

> **Can a Sabate-style neural-LV calibration fit one SPX date sensibly, and where is any remaining error coming from?**

### Core choices retained

- Same cleaned SPX universe as the current classical LV/Heston notebooks.
- 25 OTM calibration quotes per expiry/settlement slice.
- Shared neural local volatility $\sigma_\theta(t,X)$, with $X=S/F_0$.
- Sabate/Gierjatowicz tamed-Euler simulation.
- Alternating SDE and control-variate optimisation.
- **One control-variate network per maturity interval**, matching Sabate's `Net_timegrid` idea.
- Each control-variate network has one hedge output per selected calibration option.
- Sabate's initialisation is followed exactly: Xavier-normal **weights only**; biases are left at PyTorch's default initial values.
- Raw Monte Carlo pricing error decides the best SDE checkpoint. Control-variate diagnostics are reported separately.
- No plotting and no correlation calculations, to reduce the native-kernel crashes seen in earlier versions.

### Important scope

This is still a **diagnostic**, not a Sabate-scale final run. It uses materially more training than v3's quick test, but remains far below 40,000 paths × 20 SGD iterations × hundreds of SDE epochs.


## 1. Setup and one fixed diagnostic configuration


In [1]:
from pathlib import Path
from dataclasses import dataclass
import copy, gc, json, math, os, random, time, warnings

# Conservative native-threading limits before NumPy / PyTorch imports.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from tqdm.auto import tqdm
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch, "xpu"):
        try:
            if torch.xpu.is_available():
                return torch.device("xpu")
        except Exception:
            pass
    return torch.device("cpu")

DEVICE = choose_device()

if DEVICE.type == "cpu":
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

# ----- Fixed v4 diagnostic settings -----
EXAMPLE_DATE = "2024-03-01"
FORCE_RETRAIN = False

EPOCHS = 160
PATHS_PER_STEP = 1_024
STEPS_PER_EPOCH = 6
QUOTES_PER_MATURITY_PER_STEP = 8

FIXED_RAW_EVAL_PATHS = 4_096
CV_EVAL_PATHS = 2_048
CV_EVAL_CONTRACTS = 64
EVAL_EVERY_SDE_EPOCHS = 5
CV_EVAL_EVERY_RAW_CHECKS = 2
PATIENCE_RAW_CHECKS = 8
HEDGE_REFINEMENT_EPOCHS = 10

FINAL_MC_PATHS = 20_000
FINAL_PATH_CHUNK = 5_000
MAX_WALLCLOCK_HOURS = 3.0

# Sabate/repository architecture/training constants.
SDE_HIDDEN_LAYERS = 4
SDE_WIDTH = 50
HEDGE_HIDDEN_LAYERS = 3
HEDGE_WIDTH = 30
LEARNING_RATE = 1e-3
SDE_GRAD_CLIP = 5.0
HEDGE_GRAD_CLIP = 3.0
BASE_STEPS_PER_YEAR = 96

MAX_QUOTES_PER_SLICE = 25
DELTA_BUMP_SPX_POINTS = 0.50

CACHE_DIR = Path("neural_lv_v4_cache")
CACHE_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("PyTorch threads:", torch.get_num_threads())
print(
    f"Training: {EPOCHS} alternating epochs, "
    f"{STEPS_PER_EPOCH} steps/epoch, {PATHS_PER_STEP:,} paths/step"
)
print("Selected calibration cap:", MAX_QUOTES_PER_SLICE, "per expiry/settlement slice")


Device: cpu
PyTorch threads: 1
Training: 160 alternating epochs, 6 steps/epoch, 1,024 paths/step
Selected calibration cap: 25 per expiry/settlement slice


## 2. Load and clean the OptionMetrics data

The cleaning logic is kept aligned with the LV/Heston notebooks:

- SPX (`secid=108105`);
- European, standard 100-multiplier, unadjusted contracts;
- DTE 7–365;
- valid bid/ask and implied volatility;
- $0.05\le |\Delta|\le0.95$;
- relative spread $\le25\%$;
- $|\log(K/F)|\le0.30$;
- elementary European price bounds.

Calibration uses one OTM observation per strike and requires at least 8 observations per expiry/settlement slice.


In [2]:
def find_file(pattern):
    search_dirs = [Path("."), Path("/mnt/data")]
    found = []
    for folder in search_dirs:
        found.extend(sorted(folder.glob(pattern)))
    if not found:
        raise FileNotFoundError(f"Could not find a file matching {pattern!r}")
    return found[0]


FILES = {
    "options": find_file("one_month_spx_option_price*.csv"),
    "forwards": find_file("one_month_spx_forward_price*.csv"),
    "spot": find_file("one_month_spx_security_price*.csv"),
    "rates": find_file("one_month_spx_zero_coupon_yield_curve*.csv"),
}

for name, path in FILES.items():
    print(f"{name:8s}: {path}")

options_raw = pd.read_csv(FILES["options"], low_memory=False)
forwards_raw = pd.read_csv(FILES["forwards"], low_memory=False)
spot_raw = pd.read_csv(FILES["spot"], low_memory=False)
rates_raw = pd.read_csv(FILES["rates"], low_memory=False)



SECID = 108105
MIN_DTE, MAX_DTE = 7, 365
MAX_ABS_LOG_MONEYNESS = 0.30
MIN_ABS_DELTA, MAX_ABS_DELTA = 0.05, 0.95
MAX_RELATIVE_SPREAD = 0.25
ATM_TIGHTER_SIDE_BAND = 0.05
MIN_POINTS_PER_SLICE = 8


def prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw):
    options = options_raw.copy()
    options["date"] = pd.to_datetime(options["date"])
    options["exdate"] = pd.to_datetime(options["exdate"])
    options["K"] = options["strike_price"] / 1000.0
    options["T_days"] = (options["exdate"] - options["date"]).dt.days
    options["T"] = options["T_days"] / 365.0
    options["bid"] = pd.to_numeric(options["best_bid"], errors="coerce")
    options["ask"] = pd.to_numeric(options["best_offer"], errors="coerce")
    options["mid"] = 0.5 * (options["bid"] + options["ask"])
    options["spread"] = options["ask"] - options["bid"]
    options["relative_spread"] = options["spread"] / options["mid"].replace(0, np.nan)
    options["iv_market"] = pd.to_numeric(options["impl_volatility"], errors="coerce")
    options["delta_market"] = pd.to_numeric(options["delta"], errors="coerce")
    options["abs_delta_market"] = options["delta_market"].abs()

    mask = (
        (options["secid"] == SECID)
        & (options["exercise_style"] == "E")
        & (options["contract_size"] == 100)
        & (options["cfadj"] == 1)
        & options["cp_flag"].isin(["C", "P"])
        & (options["bid"] >= 0)
        & (options["ask"] > options["bid"])
        & (options["mid"] > 0)
        & options["T_days"].between(MIN_DTE, MAX_DTE)
        & options["iv_market"].between(1e-4, 5.0)
        & options["abs_delta_market"].between(MIN_ABS_DELTA, MAX_ABS_DELTA)
        & (options["relative_spread"] <= MAX_RELATIVE_SPREAD)
    )
    options = options.loc[mask].copy()

    spot = spot_raw.loc[spot_raw["secid"] == SECID, ["date", "close"]].copy()
    spot["date"] = pd.to_datetime(spot["date"])
    spot = spot.rename(columns={"close": "S"}).drop_duplicates("date")

    forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
    forwards["date"] = pd.to_datetime(forwards["date"])
    forwards["expiration"] = pd.to_datetime(forwards["expiration"])
    forwards = forwards.rename(columns={
        "expiration": "exdate",
        "AMSettlement": "am_settlement",
        "ForwardPrice": "F",
    })
    forwards = forwards[
        ["date", "exdate", "am_settlement", "F"]
    ].drop_duplicates(["date", "exdate", "am_settlement"])

    rates = rates_raw.copy()
    rates["date"] = pd.to_datetime(rates["date"])
    rates["days"] = pd.to_numeric(rates["days"], errors="coerce")
    rates["rate_decimal"] = pd.to_numeric(rates["rate"], errors="coerce") / 100.0
    rate_curves = {
        d: (g["days"].to_numpy(float), g["rate_decimal"].to_numpy(float))
        for d, g in rates.dropna().sort_values(["date", "days"]).groupby("date")
    }

    options = options.merge(spot, on="date", how="left").merge(
        forwards,
        on=["date", "exdate", "am_settlement"],
        how="left",
        validate="many_to_one",
    )
    options["r"] = [
        float(np.interp(n, *rate_curves[d])) if d in rate_curves else np.nan
        for d, n in zip(options["date"], options["T_days"])
    ]
    options["D"] = np.exp(-options["r"] * options["T"])
    options["carry"] = np.log(options["F"] / options["S"]) / options["T"]
    options["q"] = options["r"] - options["carry"]
    options["k"] = np.log(options["K"] / options["F"])

    options = options.dropna(subset=["S", "F", "r", "D", "carry", "k"])
    options = options.loc[options["k"].abs() <= MAX_ABS_LOG_MONEYNESS].copy()

    call_lb = options["D"] * np.maximum(options["F"] - options["K"], 0.0)
    call_ub = options["D"] * options["F"]
    put_lb = options["D"] * np.maximum(options["K"] - options["F"], 0.0)
    put_ub = options["D"] * options["K"]
    lb = np.where(options["cp_flag"].eq("C"), call_lb, put_lb)
    ub = np.where(options["cp_flag"].eq("C"), call_ub, put_ub)
    options = options.loc[
        (options["mid"] >= lb - 1e-6) & (options["mid"] <= ub + 1e-6)
    ].copy()

    return options.sort_values(
        ["date", "exdate", "am_settlement", "K", "cp_flag"]
    ).reset_index(drop=True)


def build_otm_surface(quotes):
    idx = [
        "date", "exdate", "am_settlement", "K", "T", "S", "F",
        "r", "q", "carry", "D", "k"
    ]
    vals = [
        "mid", "bid", "ask", "spread", "relative_spread",
        "iv_market", "delta_market", "optionid"
    ]

    wide = quotes.pivot_table(
        index=idx, columns="cp_flag", values=vals, aggfunc="first"
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    strict_call = wide["K"] >= wide["F"]
    near = wide["k"].abs() <= ATM_TIGHTER_SIDE_BAND
    have_c = wide.get("iv_market_C", pd.Series(np.nan, index=wide.index)).notna()
    have_p = wide.get("iv_market_P", pd.Series(np.nan, index=wide.index)).notna()
    c_spread = wide.get("spread_C", pd.Series(np.inf, index=wide.index))
    p_spread = wide.get("spread_P", pd.Series(np.inf, index=wide.index))

    choose_call = strict_call.copy()
    both_near = near & have_c & have_p
    choose_call = np.where(both_near, c_spread <= p_spread, choose_call)
    wide["cp_otm"] = np.where(choose_call, "C", "P")

    for base in vals:
        c = wide.get(f"{base}_C", pd.Series(np.nan, index=wide.index))
        p = wide.get(f"{base}_P", pd.Series(np.nan, index=wide.index))
        wide[f"{base}_otm"] = np.where(choose_call, c, p)

    wide = wide.dropna(subset=["iv_market_otm", "mid_otm", "optionid_otm"]).copy()

    sizes = wide.groupby(["date", "exdate", "am_settlement"]).size()
    valid = sizes[sizes >= MIN_POINTS_PER_SLICE].index

    return (
        wide.set_index(["date", "exdate", "am_settlement"])
        .loc[valid]
        .reset_index()
        .sort_values(["date", "T", "k"])
        .reset_index(drop=True)
    )


quotes_all = prepare_clean_data(options_raw, forwards_raw, spot_raw, rates_raw)
otm = build_otm_surface(quotes_all)

print("Complete cleaned quotes:", len(quotes_all))
print("OTM calibration observations:", len(otm))
print("Trading dates:", otm["date"].nunique())
print("Expiry/settlement slices:", otm.groupby(["date", "exdate", "am_settlement"]).ngroups)


options : one_month_spx_option_price.csv
forwards: one_month_spx_forward_price.csv
spot    : one_month_spx_security_price.csv
rates   : one_month_spx_zero_coupon_yield_curve.csv
Complete cleaned quotes: 192031
OTM calibration observations: 91930
Trading dates: 20
Expiry/settlement slices: 809


## 3. Curves and the 25-per-slice calibration set


In [3]:
@dataclass
class DailyMarketCurve:
    date: pd.Timestamp
    am_settlement: int
    spot: float
    T_nodes: np.ndarray
    logD_nodes: np.ndarray
    logF_nodes: np.ndarray

    def __post_init__(self):
        self._logD = PchipInterpolator(self.T_nodes, self.logD_nodes, extrapolate=True)
        self._logF = PchipInterpolator(self.T_nodes, self.logF_nodes, extrapolate=True)

    def log_discount(self, t):
        return self._logD(np.maximum(np.asarray(t, float), 0.0))

    def log_forward_ratio(self, t):
        return self._logF(np.maximum(np.asarray(t, float), 0.0))

    def discount(self, t0, t1):
        return np.exp(self.log_discount(t1) - self.log_discount(t0))

    def forward(self, t):
        return self.spot * np.exp(self.log_forward_ratio(t))

    def q_integral(self, t0, t1):
        return (
            -(self.log_discount(t1) - self.log_discount(t0))
            -(self.log_forward_ratio(t1) - self.log_forward_ratio(t0))
        )


class CurveBook:
    def __init__(self, spot_raw, forwards_raw, rates_raw):
        self.spot = spot_raw.loc[
            spot_raw["secid"] == SECID, ["date", "close"]
        ].copy()
        self.spot["date"] = pd.to_datetime(self.spot["date"])
        self.spot = self.spot.set_index("date")["close"].to_dict()

        self.forwards = forwards_raw.loc[forwards_raw["secid"] == SECID].copy()
        self.forwards["date"] = pd.to_datetime(self.forwards["date"])
        self.forwards["expiration"] = pd.to_datetime(self.forwards["expiration"])

        self.rates = rates_raw.copy()
        self.rates["date"] = pd.to_datetime(self.rates["date"])
        self.rates["rate_decimal"] = pd.to_numeric(
            self.rates["rate"], errors="coerce"
        ) / 100.0
        self._cache = {}

    def get(self, date, am_settlement):
        date = pd.Timestamp(date)
        am = int(am_settlement)
        key = (date, am)
        if key in self._cache:
            return self._cache[key]

        S0 = float(self.spot[date])

        rg = self.rates.loc[
            self.rates["date"] == date, ["days", "rate_decimal"]
        ].dropna().sort_values("days")
        T_r = rg["days"].to_numpy(float) / 365.0
        logD = -rg["rate_decimal"].to_numpy(float) * T_r

        fg = self.forwards.loc[
            (self.forwards["date"] == date)
            & (self.forwards["AMSettlement"] == am),
            ["expiration", "ForwardPrice"],
        ].copy()
        if len(fg) < 2:
            fg = self.forwards.loc[
                self.forwards["date"] == date,
                ["expiration", "ForwardPrice"],
            ].copy()

        fg["T"] = (fg["expiration"] - date).dt.days / 365.0
        fg = (
            fg.loc[(fg["T"] >= 0) & (fg["ForwardPrice"] > 0)]
            .groupby("T", as_index=False)["ForwardPrice"]
            .mean()
            .sort_values("T")
        )
        T_f = fg["T"].to_numpy(float)
        logF = np.log(fg["ForwardPrice"].to_numpy(float) / S0)
        positive = T_f > 0
        T_f, logF = T_f[positive], logF[positive]

        T_all = np.unique(np.concatenate([[0.0], T_r, T_f]))
        logD_all = np.interp(T_all, np.r_[0.0, T_r], np.r_[0.0, logD])
        logF_all = np.interp(T_all, np.r_[0.0, T_f], np.r_[0.0, logF])

        curve = DailyMarketCurve(date, am, S0, T_all, logD_all, logF_all)
        self._cache[key] = curve
        return curve


curve_book = CurveBook(spot_raw, forwards_raw, rates_raw)



def sample_calibration_day(day, max_per_slice=MAX_QUOTES_PER_SLICE):
    pieces = []
    for _, g in day.groupby(["exdate", "am_settlement"]):
        g = g.sort_values("k")
        if len(g) <= max_per_slice:
            pieces.append(g)
        else:
            pos = np.unique(
                np.round(np.linspace(0, len(g)-1, max_per_slice)).astype(int)
            )
            pieces.append(g.iloc[pos])
    return pd.concat(pieces, ignore_index=True)

def prepare_selected_calibration(date):
    day = otm.loc[otm["date"] == pd.Timestamp(date)].copy()
    selected = sample_calibration_day(day).rename(columns={
        "mid_otm": "target_mid",
        "cp_otm": "target_cp",
        "optionid_otm": "optionid",
    }).reset_index(drop=True)
    selected["contract_id"] = np.arange(len(selected), dtype=int)
    return selected

selected_example = prepare_selected_calibration(EXAMPLE_DATE)
print("Selected contracts:", len(selected_example))
print("Distinct maturity times:", selected_example["T"].nunique())
print("Expiry/settlement slices:",
      selected_example.groupby(["exdate","am_settlement"]).ngroups)


Selected contracts: 997
Distinct maturity times: 36
Expiry/settlement slices: 40


## 4. Shared neural LV and Sabate-style interval control variates

The SDE coefficient is shared:
$$
dX_t=X_t\sigma_\theta(t,X_t)\,dW_t,\qquad X_0=1.
$$

The **control variate is not shared through time**. Following Sabate's `Net_timegrid`, there is one hedge network on each maturity interval. Each interval network has one output coordinate for every selected calibration option.

For computational efficiency, an SGD mini-batch only materialises the output coordinates it needs. This is algebraically the same as taking the corresponding coordinates from Sabate's full output vector.

The initialisation below deliberately does **not** zero biases. Sabate's `init_weights` overwrites only `nn.Linear.weight`.


In [4]:
class FeedForwardTrunk(nn.Module):
    def __init__(self, input_dim, hidden_layers, width):
        super().__init__()
        layers, d = [], input_dim
        for _ in range(hidden_layers):
            layers += [nn.Linear(d, width), nn.ReLU()]
            d = width
        self.net = nn.Sequential(*layers)
        self.output_dim = width

    def forward(self, x):
        return self.net(x)

def sabate_weight_initialisation(module):
    # Exact repository behaviour: change weights only, leave biases untouched.
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight.data, gain=1.5)

class SharedNeuralLocalVol(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, SDE_HIDDEN_LAYERS, SDE_WIDTH)
        self.out = nn.Sequential(nn.Linear(SDE_WIDTH, 1), nn.Softplus())
        self.apply(sabate_weight_initialisation)

    def forward(self, t, x):
        if x.ndim == 1:
            x = x[:, None]
        if not torch.is_tensor(t):
            t = torch.full_like(x, float(t))
        elif t.ndim == 0:
            t = torch.full_like(x, float(t.item()))
        elif t.ndim == 1:
            t = t[:, None]
        return self.out(self.trunk(torch.cat([t, x], dim=1)))

class PerOptionHedgeNet(nn.Module):
    def __init__(self, n_options):
        super().__init__()
        self.trunk = FeedForwardTrunk(2, HEDGE_HIDDEN_LAYERS, HEDGE_WIDTH)
        self.output = nn.Linear(HEDGE_WIDTH, int(n_options))
        self.apply(sabate_weight_initialisation)

    def forward_ids(self, t, x, option_ids):
        if x.ndim == 1:
            x = x[:, None]
        if not torch.is_tensor(t):
            t = torch.full_like(x, float(t))
        elif t.ndim == 0:
            t = torch.full_like(x, float(t.item()))
        elif t.ndim == 1:
            t = t[:, None]

        h = self.trunk(torch.cat([t, x], dim=1))
        ids = torch.as_tensor(option_ids, dtype=torch.long, device=h.device)
        W = self.output.weight[ids]
        b = self.output.bias[ids]
        return h @ W.T + b[None, :]

class IntervalHedgeBank(nn.Module):
    def __init__(self, maturity_boundaries, n_options):
        super().__init__()
        self.boundaries = np.asarray(maturity_boundaries, float)
        self.nets = nn.ModuleList([
            PerOptionHedgeNet(n_options) for _ in self.boundaries
        ])

    def interval_index(self, t):
        tt = float(t.reshape(-1)[0].item()) if torch.is_tensor(t) else float(t)
        idx = int(np.searchsorted(self.boundaries, tt + 1e-7, side="right"))
        return min(idx, len(self.nets)-1)

    def forward_ids(self, t, x, option_ids):
        return self.nets[self.interval_index(t)].forward_ids(t, x, option_ids)

def set_requires_grad(module, value):
    for p in module.parameters():
        p.requires_grad_(value)


## 5. Time grid, calibration context and Monte Carlo estimators


In [5]:
def build_time_grid(maturities, steps_per_year=BASE_STEPS_PER_YEAR):
    maturities = np.sort(np.unique(np.asarray(maturities, float)))
    max_T = float(maturities[-1])
    base = np.arange(0.0, max_T + 0.5/steps_per_year, 1.0/steps_per_year)
    base = base[base <= max_T + 1e-12]
    grid = np.unique(np.r_[0.0, base, maturities, max_T])
    grid.sort()
    return grid

def nearest_grid_indices(grid, times):
    grid, times = np.asarray(grid), np.asarray(times)
    idx = np.clip(np.searchsorted(grid, times), 0, len(grid)-1)
    left = np.maximum(idx-1, 0)
    idx = np.where(np.abs(grid[left]-times) < np.abs(grid[idx]-times), left, idx)
    if np.max(np.abs(grid[idx]-times)) > 1e-10:
        raise RuntimeError("A listed maturity was not represented exactly on the simulation grid.")
    return idx

@dataclass
class CalibrationContext:
    date: pd.Timestamp
    spot: float
    contracts: pd.DataFrame
    maturity_boundaries: np.ndarray
    grid_np: np.ndarray
    grid: torch.Tensor
    maturity_index: torch.Tensor
    stage_index: np.ndarray
    K: torch.Tensor
    F_T: torch.Tensor
    D_T: torch.Tensor
    target: torch.Tensor
    cp_sign: torch.Tensor
    am: torch.Tensor
    F_grid: torch.Tensor
    D_grid: torch.Tensor

def build_calibration_context(date):
    contracts = prepare_selected_calibration(date)
    date = pd.Timestamp(date)
    boundaries = np.sort(contracts["T"].unique())
    grid_np = build_time_grid(boundaries)
    maturity_idx = nearest_grid_indices(grid_np, contracts["T"].to_numpy(float))
    stage_map = {round(float(T),12): i for i,T in enumerate(boundaries)}
    stage_index = np.array(
        [stage_map[round(float(T),12)] for T in contracts["T"]], dtype=int
    )

    S0 = float(contracts["S"].iloc[0])
    F_grid = np.empty((2, len(grid_np)))
    D_grid = np.empty((2, len(grid_np)))
    for am in [0,1]:
        curve = curve_book.get(date, am)
        F_grid[am] = curve.forward(grid_np)
        D_grid[am] = curve.discount(0.0, grid_np)

    return CalibrationContext(
        date=date,
        spot=S0,
        contracts=contracts,
        maturity_boundaries=boundaries,
        grid_np=grid_np,
        grid=torch.tensor(grid_np, dtype=torch.float32, device=DEVICE),
        maturity_index=torch.tensor(maturity_idx, dtype=torch.long, device=DEVICE),
        stage_index=stage_index,
        K=torch.tensor(contracts["K"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        F_T=torch.tensor(contracts["F"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        D_T=torch.tensor(contracts["D"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        target=torch.tensor(contracts["target_mid"].to_numpy(float), dtype=torch.float32, device=DEVICE),
        cp_sign=torch.tensor(
            np.where(contracts["target_cp"].eq("C"),1.0,-1.0),
            dtype=torch.float32, device=DEVICE
        ),
        am=torch.tensor(
            contracts["am_settlement"].to_numpy(int),
            dtype=torch.long, device=DEVICE
        ),
        F_grid=torch.tensor(F_grid, dtype=torch.float32, device=DEVICE),
        D_grid=torch.tensor(D_grid, dtype=torch.float32, device=DEVICE),
    )

def antithetic_normals(total_paths, n_steps, seed):
    half = (total_paths+1)//2
    gen = torch.Generator(device="cpu")
    gen.manual_seed(int(seed))
    z = torch.randn(half, n_steps, generator=gen, dtype=torch.float32)
    return torch.cat([z,-z], dim=0)[:total_paths].to(DEVICE)

def empirical_payoff_mean(samples, strike_norm, cp_flag):
    x = np.sort(np.asarray(samples,float))
    n = len(x)
    prefix = np.r_[0.0, np.cumsum(x)]
    total = prefix[-1]
    k = np.asarray(strike_norm,float)
    idx = np.searchsorted(x,k,side="right")
    below = prefix[idx]
    above = total-below
    call = (above-k*(n-idx))/n
    put = (k*idx-below)/n
    return np.where(np.asarray(cp_flag)=="C",call,put)

def simulate_estimators(model, hedge, ctx, contract_ids, z, phase):
    ids_np = np.asarray(contract_ids,dtype=int)
    ids = torch.as_tensor(ids_np,dtype=torch.long,device=DEVICE)

    maturity_idx = ctx.maturity_index[ids]
    K, F_T, D_T = ctx.K[ids], ctx.F_T[ids], ctx.D_T[ids]
    cp, am = ctx.cp_sign[ids], ctx.am[ids]

    B, M = z.shape[0], len(ids_np)
    x = torch.ones((B,1),dtype=torch.float32,device=DEVICE)
    cv = torch.zeros((B,M),dtype=torch.float32,device=DEVICE)
    raw = torch.zeros((B,M),dtype=torch.float32,device=DEVICE)
    max_step = int(maturity_idx.max().item())

    for step in range(max_step):
        t = ctx.grid[step]
        dt = ctx.grid[step+1]-ctx.grid[step]
        sqrt_dt = torch.sqrt(dt)

        if phase == "hedge":
            with torch.no_grad():
                sigma = model(t,x)
        else:
            sigma = model(t,x)

        dW = sqrt_dt*z[:,step:step+1]

        active = torch.where(maturity_idx > step)[0]
        if len(active):
            active_ids = ids_np[active.detach().cpu().numpy()]
            if phase == "sde":
                with torch.no_grad():
                    h = hedge.forward_ids(t,x.detach(),active_ids)
            else:
                h = hedge.forward_ids(t,x.detach(),active_ids)

            # Sabate detach: the CV affects the sample estimator but does not
            # create an additional theta-gradient path through the hedge.
            stochastic_piece = x.detach()*sigma.detach()*dW
            discounted_forward = (
                ctx.D_grid[am[active],step]*ctx.F_grid[am[active],step]
            )
            cv[:,active] += h*stochastic_piece*discounted_forward[None,:]

        a = x*sigma
        denominator = 1.0 + torch.abs(x.detach()*sigma.detach())*sqrt_dt
        x = x + a*dW/denominator

        matured = torch.where(maturity_idx == step+1)[0]
        if len(matured):
            S_T = x*F_T[matured][None,:]
            strike = K[matured][None,:]
            call = torch.clamp(S_T-strike,min=0.0)
            put = torch.clamp(strike-S_T,min=0.0)
            raw[:,matured] = (
                torch.where((cp[matured]>0)[None,:],call,put)
                * D_T[matured][None,:]
            )

    return raw, raw-cv

class MaturityBalancedSampler:
    def __init__(self, ctx, per_maturity, seed):
        self.groups = [
            np.where(ctx.stage_index==j)[0]
            for j in range(len(ctx.maturity_boundaries))
        ]
        self.per_maturity = per_maturity
        self.rng = np.random.default_rng(seed)

    def draw(self):
        out = []
        for g in self.groups:
            n = min(self.per_maturity, len(g))
            out.extend(self.rng.choice(g,size=n,replace=False))
        return np.asarray(out,dtype=int)

@torch.no_grad()
def simulate_terminal_x(model, ctx, z):
    x = torch.ones((z.shape[0],1),dtype=torch.float32,device=DEVICE)
    stage_by_grid = {
        int(nearest_grid_indices(ctx.grid_np,[T])[0]): j
        for j,T in enumerate(ctx.maturity_boundaries)
    }
    terminal = {}

    for step in range(len(ctx.grid_np)-1):
        t = ctx.grid[step]
        dt = ctx.grid[step+1]-ctx.grid[step]
        sqrt_dt = torch.sqrt(dt)
        sigma = model(t,x)
        dW = sqrt_dt*z[:,step:step+1]
        a = x*sigma
        x = x + a*dW/(1.0+torch.abs(a)*sqrt_dt)

        if step+1 in stage_by_grid:
            terminal[stage_by_grid[step+1]] = (
                x.squeeze(1).detach().cpu().numpy().copy()
            )
    return terminal

def selected_prices_from_terminal(ctx, terminal):
    rows = []
    for stage in range(len(ctx.maturity_boundaries)):
        ids = np.where(ctx.stage_index==stage)[0]
        g = ctx.contracts.iloc[ids]
        payoff = empirical_payoff_mean(
            terminal[stage],
            g["K"].to_numpy(float)/g["F"].to_numpy(float),
            g["target_cp"].to_numpy(),
        )
        price = g["D"].to_numpy(float)*g["F"].to_numpy(float)*payoff
        tmp = g[["contract_id","optionid","T","k","target_cp","target_mid"]].copy()
        tmp["raw_model_price"] = price
        tmp["raw_error"] = price-tmp["target_mid"].to_numpy(float)
        rows.append(tmp)
    return pd.concat(rows,ignore_index=True)

def raw_selected_diagnostics(model, ctx, z_fixed):
    table = selected_prices_from_terminal(ctx, simulate_terminal_x(model,ctx,z_fixed))
    e = table["raw_error"].to_numpy(float)
    return {
        "rmse": float(np.sqrt(np.mean(e**2))),
        "mae": float(np.mean(np.abs(e))),
        "table": table,
    }

def choose_even_ids(n_total, n):
    ids = np.arange(n_total,dtype=int)
    if n is None or n >= n_total:
        return ids
    pos = np.unique(np.round(np.linspace(0,n_total-1,n)).astype(int))
    return ids[pos]

@torch.no_grad()
def cv_diagnostics(model, hedge, ctx, ids, z):
    raw, cv = simulate_estimators(model,hedge,ctx,ids,z,phase="sde")
    ids_t = torch.as_tensor(ids,dtype=torch.long,device=DEVICE)
    target = ctx.target[ids_t]
    raw_price, cv_price = raw.mean(0), cv.mean(0)
    raw_var, cv_var = raw.var(0,unbiased=False), cv.var(0,unbiased=False)
    return {
        "raw_rmse_subset": float(torch.sqrt(torch.mean((raw_price-target)**2)).cpu()),
        "cv_rmse_subset": float(torch.sqrt(torch.mean((cv_price-target)**2)).cpu()),
        "median_variance_ratio": float(
            torch.median(cv_var/torch.clamp(raw_var,min=1e-12)).cpu()
        ),
    }

calib_ctx = build_calibration_context(EXAMPLE_DATE)
print("Simulation steps:", len(calib_ctx.grid_np)-1)
print("Hedge interval networks:", len(calib_ctx.maturity_boundaries))


Simulation steps: 129
Hedge interval networks: 36


## 6. Alternating training with raw-RMSE checkpointing

Training follows Sabate's alternating logic:

- SDE epoch: minimise squared option-price error using the control-variate estimator;
- hedge epoch: minimise the pathwise estimator variance.

The best SDE checkpoint is chosen only by **raw Monte Carlo RMSE**, so a finite-sample control-variate mean cannot masquerade as a better diffusion model.


In [6]:
def checkpoint_file(date):
    return CACHE_DIR/f"shared_diag_{pd.Timestamp(date):%Y%m%d}.pt"

def train_shared(ctx):
    path = checkpoint_file(ctx.date)
    model = SharedNeuralLocalVol().to(DEVICE)
    hedge = IntervalHedgeBank(ctx.maturity_boundaries,len(ctx.contracts)).to(DEVICE)

    opt_sde = torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
    opt_hedge = torch.optim.Adam(hedge.parameters(),lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        opt_sde,milestones=[500,800],gamma=0.2
    )

    start_epoch, history, best_raw, no_improve = 0, [], np.inf, 0
    best_model_state = copy.deepcopy(model.state_dict())
    best_hedge_state = copy.deepcopy(hedge.state_dict())

    if path.exists() and not FORCE_RETRAIN:
        p = torch.load(path,map_location=DEVICE)
        if p.get("completed",False):
            model.load_state_dict(p["best_model_state"])
            hedge.load_state_dict(p["best_hedge_state"])
            print("Loaded completed v4 checkpoint:",path)
            return model,hedge,pd.DataFrame(p["history"])

        model.load_state_dict(p["model_state"])
        hedge.load_state_dict(p["hedge_state"])
        opt_sde.load_state_dict(p["opt_sde"])
        opt_hedge.load_state_dict(p["opt_hedge"])
        scheduler.load_state_dict(p["scheduler"])
        start_epoch = int(p["next_epoch"])
        history = p["history"]
        best_raw = float(p["best_raw"])
        no_improve = int(p["no_improve"])
        best_model_state = p["best_model_state"]
        best_hedge_state = p["best_hedge_state"]
        print("Resuming v4 checkpoint at epoch",start_epoch+1)

    sampler = MaturityBalancedSampler(
        ctx,QUOTES_PER_MATURITY_PER_STEP,SEED+10
    )
    z_raw = antithetic_normals(
        FIXED_RAW_EVAL_PATHS,len(ctx.grid_np)-1,SEED+100
    )
    z_cv = antithetic_normals(
        CV_EVAL_PATHS,len(ctx.grid_np)-1,SEED+200
    )
    cv_ids = choose_even_ids(len(ctx.contracts),CV_EVAL_CONTRACTS)

    sde_count = sum(1 for e in range(start_epoch) if e%2==0)
    raw_checks = 0
    step_times = []
    t0 = time.time()
    stop_reason = None

    bar = tqdm(range(start_epoch,EPOCHS),desc="Shared neural LV v4",unit="epoch")

    for epoch in bar:
        phase = "sde" if epoch%2==0 else "hedge"
        set_requires_grad(model,phase=="sde")
        set_requires_grad(hedge,phase=="hedge")
        losses = []

        for _ in range(STEPS_PER_EPOCH):
            ids = sampler.draw()
            z = torch.randn(
                PATHS_PER_STEP,len(ctx.grid_np)-1,
                dtype=torch.float32,device=DEVICE
            )
            opt_sde.zero_grad(set_to_none=True)
            opt_hedge.zero_grad(set_to_none=True)

            tic = time.time()
            raw,cv = simulate_estimators(model,hedge,ctx,ids,z,phase)
            ids_t = torch.as_tensor(ids,dtype=torch.long,device=DEVICE)

            if phase=="sde":
                loss = torch.mean(((cv.mean(0)-ctx.target[ids_t])/ctx.spot)**2)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),SDE_GRAD_CLIP)
                opt_sde.step()
            else:
                loss = torch.var(cv/ctx.spot,dim=0,unbiased=False).mean()
                loss.backward()
                nn.utils.clip_grad_norm_(hedge.parameters(),HEDGE_GRAD_CLIP)
                opt_hedge.step()

            step_times.append(time.time()-tic)
            losses.append(float(loss.detach().cpu()))
            del raw,cv,z

        scheduler.step()
        raw_diag = cv_diag = None

        if phase=="sde":
            sde_count += 1
            if sde_count%EVAL_EVERY_SDE_EPOCHS==0 or epoch==start_epoch:
                set_requires_grad(model,False)
                set_requires_grad(hedge,False)
                raw_diag = raw_selected_diagnostics(model,ctx,z_raw)
                raw_checks += 1

                if raw_diag["rmse"] < best_raw:
                    best_raw = raw_diag["rmse"]
                    no_improve = 0
                    best_model_state = copy.deepcopy(model.state_dict())
                    best_hedge_state = copy.deepcopy(hedge.state_dict())
                else:
                    no_improve += 1

                if raw_checks%CV_EVAL_EVERY_RAW_CHECKS==0:
                    cv_diag = cv_diagnostics(model,hedge,ctx,cv_ids,z_cv)

                torch.save({
                    "completed":False,
                    "next_epoch":epoch+1,
                    "model_state":model.state_dict(),
                    "hedge_state":hedge.state_dict(),
                    "opt_sde":opt_sde.state_dict(),
                    "opt_hedge":opt_hedge.state_dict(),
                    "scheduler":scheduler.state_dict(),
                    "history":history,
                    "best_raw":best_raw,
                    "no_improve":no_improve,
                    "best_model_state":best_model_state,
                    "best_hedge_state":best_hedge_state,
                },path)

                if no_improve >= PATIENCE_RAW_CHECKS:
                    stop_reason = (
                        f"raw RMSE did not improve for {PATIENCE_RAW_CHECKS} checks"
                    )

        row = {
            "epoch":epoch+1,
            "phase":phase,
            "train_loss":float(np.mean(losses)),
            "raw_rmse":np.nan if raw_diag is None else raw_diag["rmse"],
            "raw_mae":np.nan if raw_diag is None else raw_diag["mae"],
            "cv_rmse_subset":np.nan if cv_diag is None else cv_diag["cv_rmse_subset"],
            "median_variance_ratio":(
                np.nan if cv_diag is None else cv_diag["median_variance_ratio"]
            ),
            "elapsed_hours":(time.time()-t0)/3600,
        }
        history.append(row)

        post={"phase":phase,"loss":f"{row['train_loss']:.2e}"}
        if raw_diag is not None:
            post["raw_RMSE"]=f"{raw_diag['rmse']:.3f}"
        if cv_diag is not None:
            post["CV/raw var"]=f"{cv_diag['median_variance_ratio']:.2f}"
        bar.set_postfix(post)

        if len(step_times)==5:
            sec = float(np.mean(step_times))
            remaining=(EPOCHS-epoch-1)*STEPS_PER_EPOCH
            print(
                f"Measured {sec:.2f}s/optimisation step; "
                f"rough remaining training time {(sec*remaining)/3600:.2f}h."
            )

        if MAX_WALLCLOCK_HOURS is not None and (time.time()-t0)/3600 >= MAX_WALLCLOCK_HOURS:
            stop_reason=f"{MAX_WALLCLOCK_HOURS}h wall-clock budget reached"

        if stop_reason:
            print("Stopping:",stop_reason)
            break

        gc.collect()

    # Restore diffusion at the best raw-MC checkpoint.
    model.load_state_dict(best_model_state)
    hedge.load_state_dict(best_hedge_state)

    # Hedge-only refinement cannot alter pricing fit.
    if HEDGE_REFINEMENT_EPOCHS>0:
        set_requires_grad(model,False)
        set_requires_grad(hedge,True)
        refine_opt=torch.optim.Adam(hedge.parameters(),lr=LEARNING_RATE)
        for _ in tqdm(range(HEDGE_REFINEMENT_EPOCHS),desc="CV refinement",leave=False):
            for _ in range(STEPS_PER_EPOCH):
                ids=sampler.draw()
                z=torch.randn(
                    PATHS_PER_STEP,len(ctx.grid_np)-1,
                    dtype=torch.float32,device=DEVICE
                )
                refine_opt.zero_grad(set_to_none=True)
                _,cv=simulate_estimators(model,hedge,ctx,ids,z,phase="hedge")
                loss=torch.var(cv/ctx.spot,dim=0,unbiased=False).mean()
                loss.backward()
                nn.utils.clip_grad_norm_(hedge.parameters(),HEDGE_GRAD_CLIP)
                refine_opt.step()
                del cv,z

    model.eval(); hedge.eval()
    set_requires_grad(model,False); set_requires_grad(hedge,False)

    torch.save({
        "completed":True,
        "best_model_state":model.state_dict(),
        "best_hedge_state":hedge.state_dict(),
        "history":history,
        "best_raw":best_raw,
        "stop_reason":stop_reason,
    },path)

    return model,hedge,pd.DataFrame(history)

calibration_date=pd.Timestamp(EXAMPLE_DATE)
dates=sorted(pd.to_datetime(quotes_all["date"].unique()))
next_date=dates[dates.index(calibration_date)+1]

print("Calibration date:",calibration_date.date())
print("Next trading date:",next_date.date())

model,hedge,training_history=train_shared(calib_ctx)

checks=training_history.dropna(subset=["raw_rmse"])[
    ["epoch","raw_rmse","raw_mae","cv_rmse_subset","median_variance_ratio","elapsed_hours"]
]
print("\nRaw-pricing checkpoints only:")
print(checks.to_string(index=False))


Calibration date: 2024-03-01
Next trading date: 2024-03-04


Shared neural LV v4:   0%|          | 0/160 [00:00<?, ?epoch/s]

CV refinement:   0%|          | 0/10 [00:00<?, ?it/s]


Raw-pricing checkpoints only:
 epoch  raw_rmse   raw_mae  cv_rmse_subset  median_variance_ratio  elapsed_hours
     1 15.264446 12.461190             NaN                    NaN       0.000875
     9 14.569523 10.043142       13.146543               1.014721       0.009769
    19 11.968601  9.146194             NaN                    NaN       0.020608
    29 11.197893  8.043334       10.007080               0.639402       0.031182
    39 12.946197  9.541670             NaN                    NaN       0.041267
    49  8.578153  6.734286        8.230258               0.563665       0.051329
    59 10.502110  8.052582             NaN                    NaN       0.061462
    69  7.278662  5.590982        6.478005               0.514278       0.071534
    79  6.167070  5.087443             NaN                    NaN       0.081775
    89  5.636613  4.608549        6.182289               0.481220       0.094583
    99  5.886633  4.665622             NaN                    NaN       0.1083

## 7. Training sanity checks and same-day pricing

Three levels are reported:

1. selected 25-per-slice calibration contracts;
2. the full cleaned OTM strip;
3. the complete cleaned call/put chain.

The maturity-bucket table is retained because v3 showed that long maturities were the main source of underfit.


In [7]:
def tkey(t):
    return round(float(t), 12)


@torch.no_grad()
def simulate_same_day_x_samples(model, maturities, total_paths, spot, bump, chunk_paths, seed):
    grid = build_time_grid(maturities)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)
    collected = {key: [] for key in maturity_idx}

    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)
    done = 0
    bar = tqdm(total=total_paths, desc="Final same-day MC", unit="path")

    while done < total_paths:
        n = min(chunk_paths, total_paths - done)
        half = (n + 1) // 2
        z_half = torch.randn(half, len(grid)-1, generator=gen, dtype=torch.float32)
        z = torch.cat([z_half, -z_half], dim=0)[:n].to(DEVICE)
        x0 = torch.tensor(
            [1.0, (spot+bump)/spot, (spot-bump)/spot],
            dtype=torch.float32, device=DEVICE
        )
        x = x0[None, :].repeat(n, 1)

        for step in range(len(grid)-1):
            t = float(grid[step])
            dt = float(grid[step+1]-grid[step])
            sqrt_dt = math.sqrt(dt)
            xflat = x.reshape(n*3, 1)
            tflat = torch.full_like(xflat, t)
            sigma = model(tflat, xflat).reshape(n,3)
            dW = sqrt_dt * z[:, step:step+1]
            a = x * sigma
            x = x + a*dW/(1.0 + torch.abs(a)*sqrt_dt)
            idx = step+1
            if idx in index_to_keys:
                arr = x.detach().cpu().numpy()
                for key in index_to_keys[idx]:
                    collected[key].append(arr.copy())
        done += n
        bar.update(n)
        del z, x
        gc.collect()

    bar.close()
    return {k: np.concatenate(v, axis=0) for k,v in collected.items()}


def price_rows_from_same_day_samples(rows, samples_by_T, base_col="mid"):
    out = rows.copy().reset_index(drop=True)
    base_price = np.empty(len(out))
    up_price = np.empty(len(out))
    down_price = np.empty(len(out))

    for T, idx in out.groupby("T").groups.items():
        pos = np.asarray(list(idx), dtype=int)
        g = out.loc[idx]
        sample = samples_by_T[tkey(T)]
        K_over_F = g["K"].to_numpy(float) / g["F"].to_numpy(float)
        cp = g["cp_flag"].to_numpy()
        scale = g["D"].to_numpy(float) * g["F"].to_numpy(float)
        base_price[pos] = scale * empirical_payoff_mean(sample[:,0], K_over_F, cp)
        up_price[pos] = scale * empirical_payoff_mean(sample[:,1], K_over_F, cp)
        down_price[pos] = scale * empirical_payoff_mean(sample[:,2], K_over_F, cp)

    out["neural_lv_price"] = base_price
    out["neural_lv_delta"] = (up_price-down_price)/(2*DELTA_BUMP_SPX_POINTS)
    out["pricing_error"] = out["neural_lv_price"] - out[base_col]
    out["inside_spread"] = out["neural_lv_price"].between(out["bid"], out["ask"])
    return out



# Independent final training diagnostic.
z_final_diag=antithetic_normals(
    8_192,len(calib_ctx.grid_np)-1,SEED+600
)
diag=raw_selected_diagnostics(model,calib_ctx,z_final_diag)
selected_error_table=diag["table"].copy()
selected_error_table["maturity_bucket"]=pd.cut(
    selected_error_table["T"],
    bins=[0,30/365,90/365,180/365,366/365],
    labels=["7-30d","31-90d","91-180d","181-365d"],
    include_lowest=True,
)
selected_by_T=selected_error_table.groupby(
    "maturity_bucket",observed=True
)["raw_error"].agg(
    n="size",
    RMSE=lambda x:float(np.sqrt(np.mean(np.asarray(x,float)**2))),
    MAE=lambda x:float(np.mean(np.abs(np.asarray(x,float)))),
)

cv_ids=choose_even_ids(len(calib_ctx.contracts),CV_EVAL_CONTRACTS)
z_cv_final=antithetic_normals(
    CV_EVAL_PATHS,len(calib_ctx.grid_np)-1,SEED+601
)
cv_diag=cv_diagnostics(model,hedge,calib_ctx,cv_ids,z_cv_final)

print(f"Final selected-strip raw RMSE: ${diag['rmse']:.4f}")
print(f"Final selected-strip raw MAE:  ${diag['mae']:.4f}")
print(f"Median CV/raw variance ratio:  {cv_diag['median_variance_ratio']:.4f}")
print("\nSelected-strip raw error by maturity:")
print(selected_by_T.to_string())

day0_full=quotes_all.loc[
    quotes_all["date"]==calibration_date
].copy().reset_index(drop=True)
day0_otm=otm.loc[
    otm["date"]==calibration_date
].copy().reset_index(drop=True)

samples_same_day=simulate_same_day_x_samples(
    model,
    day0_full["T"].unique(),
    FINAL_MC_PATHS,
    float(day0_full["S"].iloc[0]),
    DELTA_BUMP_SPX_POINTS,
    FINAL_PATH_CHUNK,
    SEED+700,
)

same_day_full=price_rows_from_same_day_samples(
    day0_full,samples_same_day,base_col="mid"
)

otm_eval=day0_otm.rename(columns={
    "mid_otm":"mid",
    "bid_otm":"bid",
    "ask_otm":"ask",
    "cp_otm":"cp_flag",
    "optionid_otm":"optionid",
}).copy()
same_day_otm=price_rows_from_same_day_samples(
    otm_eval,samples_same_day,base_col="mid"
)

selected_ids=set(calib_ctx.contracts["optionid"].astype(int))
same_day_selected=same_day_full.loc[
    same_day_full["optionid"].astype(int).isin(selected_ids)
].copy()

def price_summary(df):
    e=df["pricing_error"].to_numpy(float)
    return {
        "n":len(df),
        "RMSE":float(np.sqrt(np.mean(e**2))),
        "MAE":float(np.mean(np.abs(e))),
        "inside_bid_ask":float(df["inside_spread"].mean()),
    }

hierarchy=pd.DataFrame({
    "selected_25_per_slice":price_summary(same_day_selected),
    "full_OTM_strip":price_summary(same_day_otm),
    "complete_clean_chain":price_summary(same_day_full),
}).T

print("\nSame-day pricing hierarchy:")
print(hierarchy.to_string())

full_loc=same_day_full.copy()
full_loc["maturity_bucket"]=pd.cut(
    full_loc["T"],
    bins=[0,30/365,90/365,180/365,366/365],
    labels=["7-30d","31-90d","91-180d","181-365d"],
    include_lowest=True,
)
full_by_T=full_loc.groupby(
    "maturity_bucket",observed=True
)["pricing_error"].agg(
    n="size",
    RMSE=lambda x:float(np.sqrt(np.mean(np.asarray(x,float)**2))),
    MAE=lambda x:float(np.mean(np.abs(np.asarray(x,float)))),
)
print("\nComplete-chain same-day error by maturity:")
print(full_by_T.to_string())

del samples_same_day,z_final_diag,z_cv_final
gc.collect()


Final selected-strip raw RMSE: $5.4114
Final selected-strip raw MAE:  $4.2870
Median CV/raw variance ratio:  0.3899

Selected-strip raw error by maturity:
                   n      RMSE       MAE
maturity_bucket                         
7-30d            225  3.179280  2.649063
31-90d           347  4.381947  3.644787
91-180d          200  6.425145  5.406659
181-365d         225  7.303556  5.920152


Final same-day MC:   0%|          | 0/20000 [00:00<?, ?path/s]


Same-day pricing hierarchy:
                            n      RMSE       MAE  inside_bid_ask
selected_25_per_slice   997.0  6.012743  4.672177        0.028084
full_OTM_strip         4304.0  6.136451  4.850949        0.028113
complete_clean_chain   9046.0  9.158498  7.379258        0.052730

Complete-chain same-day error by maturity:
                    n       RMSE       MAE
maturity_bucket                           
7-30d            1647   4.391380  3.726259
31-90d           2936   7.909379  6.672436
91-180d          3056  10.924441  9.121333
181-365d         1407  11.267112  9.346522


17

## 8. Frozen next-day pricing and the common one-day delta hedge


In [8]:
def build_evaluation_pairs(quotes):
    trading_dates = sorted(quotes["date"].unique())
    next_map = {trading_dates[i]: trading_dates[i+1] for i in range(len(trading_dates)-1)}
    left = quotes.copy()
    left["next_date"] = left["date"].map(next_map)
    cols = [
        "date","optionid","cp_flag","K","exdate","am_settlement",
        "mid","bid","ask","S","F","D","r","q","carry","T","k"
    ]
    right = quotes[cols].rename(columns={
        "date":"next_date","mid":"mid_next","bid":"bid_next","ask":"ask_next",
        "S":"S_next","F":"F_next","D":"D_next","r":"r_next","q":"q_next",
        "carry":"carry_next","T":"T_next","k":"k_next",
    })
    return left.merge(
        right,
        on=["next_date","optionid","cp_flag","K","exdate","am_settlement"],
        how="inner", validate="many_to_one"
    )


evaluation_pairs = build_evaluation_pairs(quotes_all)
transition_pairs = evaluation_pairs.loc[
    (evaluation_pairs["date"] == calibration_date)
    & (evaluation_pairs["next_date"] == next_date)
].copy().reset_index(drop=True)
print("Identical-contract pairs:", len(transition_pairs))



@torch.no_grad()
def simulate_next_day_y_samples(
    model, curve0, curve1, elapsed, maturities_remaining,
    total_paths, chunk_paths, seed
):
    grid = build_time_grid(maturities_remaining)
    maturity_idx = {
        tkey(T): int(nearest_grid_indices(grid, [T])[0])
        for T in np.unique(np.asarray(maturities_remaining, float))
    }
    index_to_keys = {}
    for key, idx in maturity_idx.items():
        index_to_keys.setdefault(idx, []).append(key)
    collected = {key: [] for key in maturity_idx}
    gen = torch.Generator(device="cpu"); gen.manual_seed(seed)
    done = 0
    bar = tqdm(total=total_paths, desc=f"Next-day MC AM={curve1.am_settlement}", unit="path")

    while done < total_paths:
        n = min(chunk_paths, total_paths-done)
        half=(n+1)//2
        zh=torch.randn(half,len(grid)-1,generator=gen,dtype=torch.float32)
        z=torch.cat([zh,-zh],dim=0)[:n].to(DEVICE)
        y=torch.ones((n,1),dtype=torch.float32,device=DEVICE)

        for step in range(len(grid)-1):
            v=float(grid[step]); dt=float(grid[step+1]-grid[step]); sqrt_dt=math.sqrt(dt)
            abs_t=elapsed+v
            Fnew=float(curve1.forward(v)); Fold=float(curve0.forward(abs_t))
            x_old=y*(Fnew/Fold)
            sigma=model(torch.full_like(y,abs_t),x_old)
            dW=sqrt_dt*z[:,step:step+1]
            a=y*sigma
            y=y+a*dW/(1.0+torch.abs(a)*sqrt_dt)
            idx=step+1
            if idx in index_to_keys:
                arr=y.squeeze(1).detach().cpu().numpy()
                for key in index_to_keys[idx]:
                    collected[key].append(arr.copy())
        done+=n; bar.update(n)
        del z,y; gc.collect()
    bar.close()
    return {k:np.concatenate(v) for k,v in collected.items()}


def price_next_day_from_samples(pairs, samples_by_am):
    out=pairs.copy().reset_index(drop=True)
    price=np.empty(len(out))
    for (am,T),idx in out.groupby(["am_settlement","T_next"]).groups.items():
        pos=np.asarray(list(idx),dtype=int); g=out.loc[idx]
        samples=samples_by_am[int(am)][tkey(T)]
        strike_norm=g["K"].to_numpy(float)/g["F_next"].to_numpy(float)
        payoff=empirical_payoff_mean(samples,strike_norm,g["cp_flag"].to_numpy())
        price[pos]=g["D_next"].to_numpy(float)*g["F_next"].to_numpy(float)*payoff
    out["model_price_next"]=price
    out["next_day_pricing_error"]=price-out["mid_next"].to_numpy(float)
    out["next_day_inside_bid_ask"]=out["model_price_next"].between(out["bid_next"],out["ask_next"])
    return out


elapsed=(next_date-calibration_date).days/365.0
samples_next_by_am={}
for am in sorted(transition_pairs["am_settlement"].unique()):
    g=transition_pairs.loc[transition_pairs["am_settlement"]==am]
    curve0=curve_book.get(calibration_date,int(am)); curve1=curve_book.get(next_date,int(am))
    samples_next_by_am[int(am)]=simulate_next_day_y_samples(
        model,curve0,curve1,elapsed,g["T_next"].unique(),
        FINAL_MC_PATHS,FINAL_PATH_CHUNK,SEED+800+int(am)
    )

next_day_results=price_next_day_from_samples(transition_pairs,samples_next_by_am)
e=next_day_results["next_day_pricing_error"].to_numpy(float)
print("Next-day n:",len(next_day_results))
print("Next-day RMSE: $",round(float(np.sqrt(np.mean(e**2))),4))
print("Next-day MAE:  $",round(float(np.mean(np.abs(e))),4))
print("Next-day inside bid-ask:",round(float(next_day_results["next_day_inside_bid_ask"].mean()),4))



delta_lookup=same_day_full[["optionid","neural_lv_delta"]].drop_duplicates("optionid")
hedge_results=next_day_results.merge(delta_lookup,on="optionid",how="left",validate="many_to_one")


def one_day_hedging_error(rows, hedge_ratio):
    rows=rows.copy().reset_index(drop=True)
    hedge_ratio=np.asarray(hedge_ratio,float)
    q_integral=np.empty(len(rows)); cash_interest=np.empty(len(rows))
    dt=(rows["next_date"].iloc[0]-rows["date"].iloc[0]).days/365.0
    for am,idx in rows.groupby("am_settlement").groups.items():
        pos=np.asarray(list(idx),dtype=int)
        curve0=curve_book.get(rows.loc[idx,"date"].iloc[0],int(am))
        q_integral[pos]=float(curve0.q_integral(0.0,dt))
        cash0=rows.loc[idx,"mid"].to_numpy(float)-hedge_ratio[pos]*rows.loc[idx,"S"].to_numpy(float)
        cash_interest[pos]=cash0*(1.0/float(curve0.discount(0.0,dt))-1.0)
    dividend_income=hedge_ratio*rows["S"].to_numpy(float)*q_integral
    return (
        rows["mid_next"].to_numpy(float)-rows["mid"].to_numpy(float)
        -hedge_ratio*(rows["S_next"].to_numpy(float)-rows["S"].to_numpy(float))
        -dividend_income-cash_interest
    )


hedge_results["delta_hedging_error"]=one_day_hedging_error(
    hedge_results,hedge_results["neural_lv_delta"]
)
hedge_results["unhedged_change"]=hedge_results["mid_next"]-hedge_results["mid"]

def hedge_summary(error,unhedged):
    error=np.asarray(error,float); unhedged=np.asarray(unhedged,float)
    mae=float(np.mean(np.abs(error))); umae=float(np.mean(np.abs(unhedged)))
    return {
        "n":len(error),"RMSE":float(np.sqrt(np.mean(error**2))),"MAE":mae,
        "bias":float(np.mean(error)),"unhedged_MAE":umae,"improvement":1-mae/umae
    }

print("Model-delta one-day hedge:")
print(pd.Series(hedge_summary(hedge_results["delta_hedging_error"],hedge_results["unhedged_change"])).to_string())



del samples_next_by_am
gc.collect()


Identical-contract pairs: 8771


Next-day MC AM=0:   0%|          | 0/20000 [00:00<?, ?path/s]

Next-day MC AM=1:   0%|          | 0/20000 [00:00<?, ?path/s]

Next-day n: 8771
Next-day RMSE: $ 8.5441
Next-day MAE:  $ 6.8902
Next-day inside bid-ask: 0.0405
Model-delta one-day hedge:
n               8771.000000
RMSE               0.993504
MAE                0.833860
bias              -0.662193
unhedged_MAE       4.059167
improvement        0.794574


17

## 9. Sabate-native learned hedge on selected contracts

This is an **additional** hedge result. The headline comparison remains the model delta on the complete identical-contract universe.

At $t=0$, the first interval hedge network is used, exactly as implied by the interval construction.


In [9]:
@torch.no_grad()
def learned_cv_hedges_selected(hedge,ctx):
    values=np.empty(len(ctx.contracts))
    x0=torch.ones((1,1),dtype=torch.float32,device=DEVICE)
    t0=torch.zeros_like(x0)
    for start in range(0,len(ctx.contracts),256):
        ids=np.arange(start,min(start+256,len(ctx.contracts)))
        values[ids]=hedge.forward_ids(t0,x0,ids).squeeze(0).cpu().numpy()
    return values

selected_hedges=calib_ctx.contracts[["optionid","contract_id"]].copy()
selected_hedges["learned_cv_hedge"]=learned_cv_hedges_selected(
    hedge,calib_ctx
)

selected_pairs=hedge_results.merge(
    selected_hedges[["optionid","learned_cv_hedge"]],
    on="optionid",how="inner",validate="many_to_one"
).reset_index(drop=True)

selected_pairs["cv_hedging_error"]=one_day_hedging_error(
    selected_pairs,selected_pairs["learned_cv_hedge"]
)
selected_pairs["delta_hedging_error_selected"]=one_day_hedging_error(
    selected_pairs,selected_pairs["neural_lv_delta"]
)

comparison=pd.DataFrame({
    "model_delta_selected":hedge_summary(
        selected_pairs["delta_hedging_error_selected"],
        selected_pairs["unhedged_change"]
    ),
    "learned_CV_selected":hedge_summary(
        selected_pairs["cv_hedging_error"],
        selected_pairs["unhedged_change"]
    ),
}).T

print("Selected-contract hedge comparison:")
print(comparison.to_string())


Selected-contract hedge comparison:
                          n      RMSE       MAE      bias  unhedged_MAE  improvement
model_delta_selected  950.0  0.833235  0.734836 -0.661474      1.613947     0.544696
learned_CV_selected   950.0  0.814327  0.700228 -0.622768      1.613947     0.566140


## 10. Compact final summary and exports


In [10]:
final_summary={
    "selected_same_day_RMSE":float(hierarchy.loc["selected_25_per_slice","RMSE"]),
    "full_OTM_same_day_RMSE":float(hierarchy.loc["full_OTM_strip","RMSE"]),
    "complete_chain_same_day_RMSE":float(hierarchy.loc["complete_clean_chain","RMSE"]),
    "next_day_RMSE":float(np.sqrt(np.mean(
        next_day_results["next_day_pricing_error"].to_numpy(float)**2
    ))),
    "next_day_MAE":float(np.mean(np.abs(
        next_day_results["next_day_pricing_error"].to_numpy(float)
    ))),
    "delta_hedge_RMSE":float(np.sqrt(np.mean(
        hedge_results["delta_hedging_error"].to_numpy(float)**2
    ))),
    "delta_hedge_MAE":float(np.mean(np.abs(
        hedge_results["delta_hedging_error"].to_numpy(float)
    ))),
    "cv_selected_hedge_RMSE":float(np.sqrt(np.mean(
        selected_pairs["cv_hedging_error"].to_numpy(float)**2
    ))),
    "cv_median_variance_ratio":float(cv_diag["median_variance_ratio"]),
}

print(json.dumps(final_summary,indent=2))

stem=f"{calibration_date:%Y%m%d}_shared_diagnostic_v4"
pd.DataFrame([final_summary]).to_csv(
    CACHE_DIR/f"summary_{stem}.csv",index=False
)
hierarchy.to_csv(CACHE_DIR/f"same_day_hierarchy_{stem}.csv")
selected_by_T.to_csv(CACHE_DIR/f"selected_maturity_error_{stem}.csv")
full_by_T.to_csv(CACHE_DIR/f"full_chain_maturity_error_{stem}.csv")
training_history.to_csv(CACHE_DIR/f"training_history_{stem}.csv",index=False)

print("Saved compact diagnostics under:",CACHE_DIR.resolve())


{
  "selected_same_day_RMSE": 6.012743387015091,
  "full_OTM_same_day_RMSE": 6.136450957911139,
  "complete_chain_same_day_RMSE": 9.158498268045552,
  "next_day_RMSE": 8.544074476349946,
  "next_day_MAE": 6.890158337870714,
  "delta_hedge_RMSE": 0.9935037238872766,
  "delta_hedge_MAE": 0.8338595092382879,
  "cv_selected_hedge_RMSE": 0.8143269836015439,
  "cv_median_variance_ratio": 0.3899247646331787
}
Saved compact diagnostics under: C:\Users\Startklar\ICL\NDEs\Codes\NSDE\neural_lv_v4_cache


### What counts as a successful diagnostic?

Do **not** demand Sabate's synthetic $10^{-5}$-scale calibration error from noisy market mids.

The useful questions are:

1. Is selected-strip **raw** RMSE still falling with additional SDE optimisation?
2. Is full-OTM RMSE close to selected-strip RMSE?
3. Does the 181–365d bucket improve materially from the v3 underfit?
4. Is the median control-variate variance ratio comfortably below 1?
5. After a convincing same-day fit, does frozen next-day pricing remain competitive?

If selected-strip raw RMSE still stalls around \$10–11 after this larger shared diagnostic, the next experiment should be the original Sabate piecewise-$\sigma$ architecture rather than simply multiplying the shared run time again.
